# Advanced AI Model Endpoint API Guide

This notebook provides a guide with examples and best practices for how to use AI model endpoints. You'll learn how to authenticate, configure the API client, make requests, handle responses, and implement error handling.

**Prerequisites:**
- An AI model endpoint already deployed 
- API key for the deployed endpoint 
- URL for the deployed endpoint
- Deployment name for the deployed endpoint
- Python 3.8 or higher
- Basic knowledge of Python and REST APIs

## Section 1: Set Up Credentials and Environment Variables

Configure the endpoint credentials for authentication, including the API key, endpoint URL, and model deployment name.

In [ ]:
ENDPOINT_URL = "URL" # <-----------Replace with your API endpoint URL
API_KEY = "KEY" # <----------------Replace with your authentication key (i.e., token)
DEPLOYMENT_NAME = "NAME" # <-------Replace with the deployment name, e.g., "gpt-4-34edfg"

# Examples:
# ENDPOINT_URL = "https://gpt-5-miniabc123.cognitiveservices.azure.com/openai/deployments/gpt-5-mini-abc123/chat/completions?api-version=2025-04-01-preview"
# API_KEY = "2O8UHRfD........XJ3w3AAAAACOGCTJp"
# DEPLOYMENT_NAME = "gpt-5-mini-abc123"

## Section 2: Install Required Libraries

Install necessary Python libraries for interacting with API endpoints.

In [ ]:
import subprocess
import sys

# Install required packages
packages = [
    "requests",           # For HTTP requests
    "tenacity",           # For retry logic
    "aiohttp"            # For async HTTP requests
]

print("Installing required packages...")
for package in packages:
    try:
        __import__(package.replace("-", "_"))
        print(f"✓ {package} is already installed")
    except ImportError:
        print(f"Installing {package}...")
        !sudo pip install {package}
        print(f"✓ {package} installed successfully")

print("\n✓ All required packages are ready!")

## Section 3: Configure the API Client

Initialize and configure the API client with the endpoint URL and authentication credentials.

In [ ]:
import requests
import json
from typing import Dict, Any, Optional
from datetime import datetime

class AIAPIClient:
    """
    A client for interacting with AI model endpoints.
    """
    
    def __init__(self, endpoint: str, api_key: str, model_name: str):
        """
        Initialize the API client.
        
        Args:
            endpoint: The AI endpoint URL
            api_key: The API key for authentication
            model_name: The name of the model deployment
        """
        self.endpoint = endpoint
        self.api_key = api_key
        self.model_name = model_name
        self.headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        }

# Initialize the client
client = AIAPIClient(
    endpoint=ENDPOINT_URL,
    api_key=API_KEY,
    model_name=DEPLOYMENT_NAME
)

print("✓ API Client configured successfully")
print(f"  Endpoint: {client.endpoint}")
print(f"  Model: {client.model_name}")

## Section 4: Make Requests to the Model Endpoint

Demonstrate how to construct and send requests to the model endpoint, including setting parameters like temperature, and prompt.

In [ ]:
# Extend the AIAPIClient class with request methods
def send_request(self, messages: list, **kwargs) -> Dict[str, Any]:
    """
    Send a request to the model endpoint.
    
    Args:
        messages: List of message dictionaries with 'role' and 'content' keys
        **kwargs: Additional parameters (temperature, top_p, etc.)
    
    Returns:
        The API response as a dictionary
    
    Example:
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Hello, how are you?"}
        ]
        response = client.send_request(messages, temperature=0.7)
    """
    url = self.endpoint
    
    # Prepare the payload
    payload = {
        "messages": messages,
        "model": self.model_name,
        "temperature": kwargs.get("temperature", 1)
    }
    
    # Remove None values
    payload = {k: v for k, v in payload.items() if v is not None}
    
    print(f"\n📤 Sending request to {url}")
    
    return requests.post(
        url,
        headers=self.headers,
        json=payload,
        timeout=30
    )

# Add the method to the client
AIAPIClient.send_request = send_request

# Example: Basic message request
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"}
]

response = client.send_request(messages, temperature=1)
if response.status_code == 200:
    response_data = response.json()
    print("\n✅ Response received:")
    print(json.dumps(response_data, indent=2))
else:
    print(f"\n❌ Request failed with status code {response.status_code}: {response.text}")

## Section 5: Handle API Responses

Parse and process the JSON responses from the API, extracting the model's output and metadata.

In [ ]:
def parse_response(self, response: requests.Response) -> Dict[str, Any]:
    """
    Parse the API response and extract relevant information.
    
    Args:
        response: The response object from the API
    
    Returns:
        A dictionary containing parsed response data
    """
    result = {
        "status_code": response.status_code,
        "success": response.status_code == 200,
        "timestamp": datetime.now().isoformat()
    }
    
    try:
        response_json = response.json()
        result["raw_response"] = response_json
        
        # Extract message content (varies by API version)
        if "choices" in response_json and len(response_json["choices"]) > 0:
            choice = response_json["choices"][0]
            result["message"] = choice.get("message", {}).get("content", "")
            # Check if the message is empty and get reasoning_content if available
            if not result["message"]:
                result["message"] = choice.get("message", {}).get("reasoning_content", "")
            result["finish_reason"] = choice.get("finish_reason", "")
        
        # Extract usage information
        if "usage" in response_json:
            result["usage"] = {
                "prompt_tokens": response_json["usage"].get("prompt_tokens", 0),
                "completion_tokens": response_json["usage"].get("completion_tokens", 0),
                "total_tokens": response_json["usage"].get("total_tokens", 0)
            }
        
    except json.JSONDecodeError:
        result["error"] = "Failed to parse response JSON"
        result["raw_text"] = response.text
    
    return result

def print_response(self, parsed_response: Dict[str, Any]) -> None:
    """Pretty print the parsed response."""
    print(f"\n📥 Response received (Status: {parsed_response['status_code']})")
    
    if parsed_response["success"]:
        print(f"\n✓ {parsed_response['message']}")
        
        if "usage" in parsed_response:
            usage = parsed_response["usage"]
            print(f"\n📊 Token Usage:")
            print(f"   Prompt tokens: {usage['prompt_tokens']}")
            print(f"   Completion tokens: {usage['completion_tokens']}")
            print(f"   Total tokens: {usage['total_tokens']}")
    else:
        print(f"\n✗ Error: {parsed_response.get('error', 'Unknown error')}")
        if "raw_text" in parsed_response:
            print(f"Response: {parsed_response['raw_text']}")

# Add methods to the client
AIAPIClient.parse_response = parse_response
AIAPIClient.print_response = print_response

# Example: Using the parsing and printing methods
parsed_response = client.parse_response(response)
client.print_response(parsed_response)

## Section 6: Implement Error Handling and Retry Logic

Implement try-except blocks and retry mechanisms to handle API errors, rate limiting, and network failures gracefully.

In [ ]:
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
    retry_if_result
)
import time

def send_request_with_retry(self, messages: list, max_retries: int = 3, **kwargs) -> Optional[Dict[str, Any]]:
    """
    Send a request with automatic retry logic.
    
    Args:
        messages: List of message dictionaries
        max_retries: Maximum number of retry attempts
        **kwargs: Additional parameters for the request
    
    Returns:
        Parsed response or None if all retries failed
    """
    
    @retry(
        stop=stop_after_attempt(max_retries),
        wait=wait_exponential(multiplier=1, min=2, max=10),
        retry=retry_if_result(lambda x: x is None),
        reraise=False
    )
    def _make_request():
        try:
            response = self.send_request(messages, **kwargs)
            
            # Handle rate limiting (429)
            if response.status_code == 429:
                print("⏳ Rate limited. Retrying...")
                return None
            
            # Handle server errors (5xx)
            if response.status_code >= 500:
                print(f"⚠️  Server error {response.status_code}. Retrying...")
                return None
            
            return self.parse_response(response)
            
        except requests.exceptions.Timeout:
            print("⏳ Request timeout. Retrying...")
            return None
        except requests.exceptions.ConnectionError:
            print("⏳ Connection error. Retrying...")
            return None
        except Exception as e:
            print(f"✗ Error: {str(e)}")
            return None
    
    return _make_request()

# Add the retry method to the client
AIAPIClient.send_request_with_retry = send_request_with_retry

# Example: Using the retry mechanism
print("Example error scenarios and automatic retry handling:")
print("""
1. Network Timeouts:
   - Automatically retries with exponential backoff
   - Max 3 attempts with 2-10 second delays

2. Rate Limiting (429):
   - Detected and triggers retry
   - Exponential backoff prevents overwhelming the API

3. Server Errors (5xx):
   - Transient server issues trigger automatic retry
   - Persistent errors are reported after max retries

4. Connection Errors:
   - Network connectivity issues trigger retry
   - Includes timeout, DNS, and connection reset errors
""")

## Section 7: Test with Sample Prompts

Test the API endpoint with various sample prompts to verify functionality and understand response behavior.

In [ ]:
# Test scenarios with different prompt types
test_prompts = [
    {
        "name": "Simple Question",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is machine learning?"}
        ],
        "params": {"temperature": 1}
    },
    {
        "name": "Code Generation",
        "messages": [
            {"role": "system", "content": "You are a Python programming expert."},
            {"role": "user", "content": "Write a function to calculate factorial of a number."}
        ],
        "params": {"temperature": 1}
    },
    {
        "name": "Creative Writing",
        "messages": [
            {"role": "system", "content": "You are a creative writer."},
            {"role": "user", "content": "Write a short story about a robot discovering friendship."}
        ],
        "params": {"temperature": 1}
    },
    {
        "name": "Multi-turn Conversation",
        "messages": [
            {"role": "system", "content": "You are a knowledgeable assistant."},
            {"role": "user", "content": "What is AI?"},
            {"role": "assistant", "content": "AI stands for Artificial Intelligence, which is the simulation of human intelligence in machines."},
            {"role": "user", "content": "What are its main applications?"}
        ],
        "params": {"temperature": 1}
    }
]

print("📝 Test Prompts Configuration:")
print("=" * 60)
for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{i}. {prompt['name']}")
    print(f"   Messages: {len(prompt['messages'])} turns")

# Function to test a single prompt
def test_prompt(client, test_case):
    """Test a single prompt and return results."""
    print(f"\n{'='*60}")
    print(f"Testing: {test_case['name']}")
    print(f"{'='*60}")
    
    try:
        # Send request with retry
        response = client.send_request_with_retry(
            messages=test_case['messages'],
            **test_case['params']
        )
        
        if response and response['success']:
            client.print_response(response)
            return True
        else:
            print("✗ Request failed")
            return False
            
    except Exception as e:
        print(f"✗ Error during test: {str(e)}")
        return False

# Note: Uncomment the code below to run actual tests
print("\n🚀 Running Test Prompts...")
for test_case in test_prompts:
    test_prompt(client, test_case)

## Section 8: Batch Process Multiple Requests

Demonstrate how to process multiple requests efficiently using loops and async operations to the Foundry model endpoint.

In [ ]:
import asyncio
import aiohttp
from typing import List
from collections import Counter

print("Batch Processing Methods Available:")
print("=" * 60)
print("\n1. Sequential Processing:")
print("   - Process requests one at a time")
print("   - Slower but simpler to debug")
print("   - Good for small batches or rate-limited APIs")
print("   - Usage: results = client.batch_process_sequential(prompts)")

print("\n2. Asynchronous Processing:")
print("   - Process multiple requests concurrently")
print("   - Faster for large batches")
print("   - Requires await/async context")
print("   - Usage: results = await client.batch_process_async(prompts)")

print("\n" + "=" * 60)

def _extract_message_excerpt(response_payload: Dict[str, Any], length: int = 100) -> str:
    """Return a short excerpt from the model response."""
    if not response_payload:
        return ""

    choices = response_payload.get("choices") or []
    if not choices:
        return ""

    message = choices[0].get("message", {})
    content = message.get("content") or message.get("reasoning_content") or ""
    return content[:length]

def batch_process_sequential(self, prompts: List[str]) -> List[Dict[str, Any]]:
    """Process multiple prompts sequentially."""
    prompts_to_run = list(prompts)
    total = len(prompts_to_run)

    if not prompts_to_run:
        print("⚠️ No prompts provided for sequential batch.")
        return []

    duplicates = [p for p, count in Counter(prompts_to_run).items() if count > 1]
    if duplicates:
        print(f"⚠️ Detected {len(duplicates)} duplicated prompt(s); each instance will be processed in order.")

    print(f"\nStarting sequential batch for {total} prompt(s)...")
    results = []

    for idx, prompt in enumerate(prompts_to_run, 1):
        print(f"\n[{idx}/{total}] Processing: {prompt[:50]}...")

        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ]

        response = self.send_request_with_retry(
            messages=messages,
            temperature=1
        )

        if response:
            results.append({
                "prompt": prompt,
                "response": response,
                "success": response["success"]
            })
            print("✓ Processed successfully")
            if response.get("message"):
                print(f"   Response sample: {response['message'][:100]}...")
        else:
            results.append({
                "prompt": prompt,
                "response": None,
                "success": False
            })
            print("✗ Processing failed")

    print(f"\nSequential batch finished. {len(results)} prompt(s) processed.")
    return results

async def batch_process_async(self, prompts: List[str]) -> List[Dict[str, Any]]:
    """Process multiple prompts asynchronously using aiohttp."""
    prompts_to_run = list(prompts)
    total = len(prompts_to_run)

    if not prompts_to_run:
        print("⚠️ No prompts provided for async batch.")
        return []

    duplicates = [p for p, count in Counter(prompts_to_run).items() if count > 1]
    if duplicates:
        print(f"⚠️ Detected {len(duplicates)} duplicated prompt(s); each instance will be processed in order.")

    print(f"\nStarting async batch for {total} prompt(s)...")

    headers = {
        "Authorization": f"Bearer {self.api_key}",
        "Content-Type": "application/json"
    }

    async def send_async_request(session, prompt: str) -> Dict[str, Any]:
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ]

        payload = {
            "messages": messages,
            "model": self.model_name,
            "temperature": 1
        }

        try:
            async with session.post(
                self.endpoint,
                json=payload,
                headers=headers,
                timeout=aiohttp.ClientTimeout(total=30)
            ) as response:
                data = await response.json()
                excerpt = _extract_message_excerpt(data)
                if excerpt:
                    print(f"\n[async] {prompt[:50]}... -> {excerpt}...")
                return {
                    "prompt": prompt,
                    "status": response.status,
                    "response": data,
                    "success": response.status == 200
                }
        except Exception as e:
            print(f"\n[async] {prompt[:50]}... -> error: {e}")
            return {
                "prompt": prompt,
                "status": None,
                "response": None,
                "error": str(e),
                "success": False
            }

    async with aiohttp.ClientSession() as session:
        results = await asyncio.gather(*[send_async_request(session, prompt) for prompt in prompts_to_run])

    print(f"\nAsync batch finished. {len(results)} prompt(s) processed.")
    return results

# Attach methods to the client class
AIAPIClient.batch_process_sequential = batch_process_sequential
AIAPIClient.batch_process_async = batch_process_async

# Example prompts for batch processing
batch_prompts = [
    "Explain the theory of relativity.",
    "What are the benefits of using cloud computing?",
    "Write a Python function to reverse a string.",
    "Describe the process of photosynthesis.",
    "What is the capital of Japan?"
]

RUN_SEQUENTIAL = True
RUN_ASYNC = False

print(f"\nExecution configuration -> RUN_SEQUENTIAL={RUN_SEQUENTIAL}, RUN_ASYNC={RUN_ASYNC}")
print(f"Prompts configured: {len(batch_prompts)}")

sequential_results = []
async_results = []

if RUN_SEQUENTIAL:
    print("\nRunning Sequential Batch Processing (single pass)...")
    sequential_results = client.batch_process_sequential(batch_prompts)
    if sequential_results:
        counts = Counter(item["prompt"] for item in sequential_results)
        print("\nSequential batch summary (prompt occurrences):")
        for prompt_text, count in counts.items():
            print(f"  - {prompt_text[:60]}... -> {count} time(s)")
else:
    print("\nSkipping sequential batch (set RUN_SEQUENTIAL = True to enable).")

if RUN_ASYNC:
    print("\nRunning Asynchronous Batch Processing (single pass)...")

    async def _execute_async_batch():
        return await client.batch_process_async(batch_prompts)

    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None

    if loop and loop.is_running():
        async_results = await _execute_async_batch()
    else:
        async_results = asyncio.run(_execute_async_batch())

    if async_results:
        counts = Counter(item["prompt"] for item in async_results)
        print("\nAsynchronous batch summary (prompt occurrences):")
        for prompt_text, count in counts.items():
            print(f"  - {prompt_text[:60]}... -> {count} time(s)")
else:
    print("Skipping asynchronous batch (set RUN_ASYNC = True to enable).")

## Summary and Best Practices

### Key Takeaways:
1. **Error Handling**: Implement retry logic with exponential backoff
2. **Rate Limiting**: Respect API rate limits with appropriate delays
3. **Batch Processing**: Use async operations for high-volume requests
4. **Monitoring**: Track token usage to manage costs

### Best Practices:
- Implement proper logging for debugging
- Use meaningful system prompts for consistent responses
- Monitor token usage and costs
- Test with small batches before scaling
- Implement timeout and retry mechanisms
- Use connection pooling for batch operations

### Common Issues and Solutions:

| Issue | Solution |
|-------|----------|
| 401 Unauthorized | Verify API key and endpoint URL |
| 429 Rate Limit | Implement exponential backoff retry |
| 500 Server Error | Retry after delay; check service status |
| Timeout | Increase timeout value or add a max_tokens parameter |
| High Latency | Use async processing for batch requests |
